## Wine quality modelling with `lstsq`
The scope of this notebook is to compare the `lstsq` with other methods (e.g., normal equation with matrix inversion) in terms of error and the computing time needed.

In [1]:
import pandas as pd
import numpy as np
import time
import matplotlib.pyplot as plt

In [2]:
rng = np.random.default_rng(0)

### Helper functions

In [3]:
def train_test_split(X, y, test_size=0.25, seed=None):
    n = X.shape[0]
    idx = np.arange(n)
    if seed is not None:
        rng = np.random.default_rng(seed)
        rng.shuffle(idx)
    else:
        np.random.shuffle(idx)
    cut = int(n * (1 - test_size))
    train_idx = idx[:cut]
    test_idx = idx[cut:]
    
    return X[train_idx], X[test_idx], y[train_idx], y[test_idx]

In [4]:
def standardise_training_data(training_data, test_data):
    """
    Standardise the training data so that each feature column
    has a mean of 0 and a standard deviation of 1.

    Test data is scaled using the same mean and standard 
    deviation as the training data.

    Args:
        training_data (np.ndarray): Training data matrix, shape (n, d)
        test_data (np.ndarray, optional): Test data matrix, shape (m, d)

    Returns:
        scaled_train (np.ndarray): Standardised training data, shape (n, d)
        scaled_test (np.ndarray or None): Standardised test data, shape (m, d)
        std (np.ndarray): Original standard deviations, shape (d,)
        mu (np.ndarray): Original means, shape (d,)
    
    """
        
    # Compute mean and standard deviation for each feature column
    mu = np.mean(training_data, axis=0)
    std = np.std(training_data, axis=0)
    
    # Standardize the training data
    scaled_train = (training_data - mu) / std

    # Standardize test data using training mean and std
    scaled_test = (test_data - mu) / std
    
    return scaled_train, scaled_test, std, mu

In [5]:
def build_polynomial_features(X, degree):
    """
    Build a polynomial design matrix Φ.
    Note: Polynomial regression = linear regression on nonlinear features.

    """
    
    Phi = np.ones((X.shape[0], degree + 1))
    for d in range(1, degree + 1):
        Phi[:, d] = X[:, 0]**d
    
    return Phi

In [6]:
def normal_equation(Phi, y_true, lam=0, method=None):
    """
    Compute the normal equation for linear regression.
    Optionally add regularisation to the normal equation.

    Args:
        Phi_mu (numpy array): Design matrix for the mean model, shape (n, d)
        y_true (numpy array): Target values, shape (n,)
        lam (float): Regularisation parameter (default: 0)
        method (str): Method used to fit the model.
            `lstsq` is default, which is triggered if no model
            is specified, as a safeguard.

    Returns:
        w (numpy array): Regression weights, shape (C,)
    
    """

    #Identity matrix, same size as the number of features (columns in Phi)
    I = np.eye(Phi.shape[1])

    #Do not regularise the bias term
    I[0, 0] = 0

    A = Phi.T @ Phi + lam * I
    b = Phi.T @ y_true

    if method=="solve":
        # Avoid using matrix inversion
        w = np.linalg.solve(A, b)
    elif method=="inv":
        w = np.linalg.inv(A) @ b
    else:
        # This should be the best one
        w = np.linalg.lstsq(Phi, y_true, rcond=None)[0]

    return w

In [7]:
def fit_polynomial(X, y, degree, lam=0, method=None):
    Phi = build_polynomial_features(X, degree)
    start = time.perf_counter()
    w = normal_equation(Phi, y, lam, method)
    end = time.perf_counter()
    time_diff = end - start
    y_pred = Phi @ w
    
    return w, Phi, y_pred, time_diff

In [8]:
def predict_polynomial(X, w, degree):
    Phi = build_polynomial_features(X, degree)
    return Phi @ w

In [9]:
def compute_rss(y, y_pred):
    """Compute residual sum of square loss."""
    return np.sum((y - y_pred) ** 2)

In [10]:
def plot_fitted_function(X, y_train, X_test, y_pred, plot_pred_as_line=True):
    """
    Plot the training data as a scatter plot and overlay the fitted function.
    The plot may compare the train data with the test fitted function, but also 
    the trian data with the trainining fitted function and so on.
    
    Args:
        X (numpy array): Training input features, shape (n,)
        y_train (numpy array): Training target values, shape (n,)
        X_test (numpy array): Test input features, shape (n,)
        y_pred (numpy array): Predicted outputs for the test inputs, shape (n,)
        plot_pred_as_line (bool): Plot the predictions as a continous line (fitted function) 
                            or scatter plot. Default is True.

    """

    plt.figure(figsize=(10, 6))
    
    # Scatter plot of the training data
    plt.scatter(X, y_train, color="blue", label="Training Data", alpha=0.8, s=15)
    
    # Plot the fitted function
    if plot_pred_as_line == True:
        # Sort X_test and corresponding mu for smooth line plotting
        sorted_indices = np.argsort(X_test.ravel())
        X_sorted = X_test.ravel()[sorted_indices]
        mu_sorted = y_pred.ravel()[sorted_indices]

        # Predicted mean as a continuous function (line plot)
        plt.plot(X_sorted, mu_sorted, color="green", label="Predicted Mean", linewidth=2)
    elif plot_pred_as_line == False:
        plt.scatter(X_test, y_pred, color="green", label="Predictions", alpha=0.8, s=15)
    else:
        raise ValueError("Invalid input for 'plot_pred_as_line'. Please select True or False.")
    
    plt.xlabel("Input Feature")
    plt.ylabel("Target Value")
    plt.title("Fitted Function vs Training Data")
    plt.legend()
    plt.show()

In [11]:
def plot_models_on_data(X, y, X_list, y_pred_list, degrees):
    """
    Plot training data and several model predictions on top of it.

    Args:
        X (np.ndarray): Training inputs
        y (np.ndarray): Training targets
        X_list (list): List of X arrays for each model (they are usually all the same)
        y_pred_list (list): List of predicted y for each model
        degrees (list): Polynomial degrees for labeling
    
    """

    plt.figure(figsize=(10, 6))

    # Plot original data
    plt.scatter(X, y, s=12, c="blue", label="Training Data", alpha=0.25)

    # Color cycle
    colors = ["red", "green", "purple", "brown", "black", "orange"]

    for i, (X_model, y_model) in enumerate(zip(X_list, y_pred_list)):
        deg = degrees[i]

        # Choose label based on degree
        if deg == 1:
            label = "Linear"
        elif deg == 2:
            label = "Quadratic"
        elif deg == 3:
            label = "Cubic"
        else:
            label = f"Degree {deg}"

        color = colors[i % len(colors)]

        # Plot predictions as scatter (just like your example)
        plt.scatter(X_model, y_model, s=10, c=color, label=label, alpha=0.85)

    plt.legend()
    plt.xlabel("X")
    plt.ylabel("y")
    plt.title("Comparing Polynomial Fits")
    plt.show()

In [12]:
def run_multiple_experiments(Xtrain, ytrain, Xtest, ytest, methods_list, degree=3):
    results = pd.DataFrame(columns=["method", "time", "train_rss", "test_rss"])

    for method in methods_list:

        # Fit polynomial model using the selected method
        w, Phi, y_pred_train, elapsed_time = fit_polynomial(
            Xtrain, ytrain, degree=degree, method=method
        )

        # Compute RSS
        train_rss = compute_rss(ytrain, y_pred_train)
        y_pred_test = predict_polynomial(Xtest, w, degree=degree)
        test_rss = compute_rss(ytest, y_pred_test)

        # Append one row to the DataFrame
        results.loc[len(results)] = {
            "method": method,
            "time": elapsed_time,
            "train_rss": train_rss,
            "test_rss": test_rss,
        }

    # Return df sorted by time
    return results.sort_values(by="time", ascending=True)

## Import data
Since the focus on this notebook is comparing `lstsq` with alternative methods, the usual data pre-processing and exploratory data analysis (EDA) steps will be skipped to avoid losing focus on the main task.

In [13]:
# Import the white wine data obtained by Cortez et al. (2009)
white_wine = pd.read_csv("data/winequality-white.csv", sep=";", index_col=False)
print("white_wine shape", white_wine.shape)

white_wine shape (4898, 12)


In [14]:
white_wine.head()

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality
0,7.0,0.27,0.36,20.7,0.045,45.0,170.0,1.0010,3.00,0.45,8.8,6
1,6.3,0.30,0.34,1.6,0.049,14.0,132.0,0.9940,3.30,0.49,9.5,6
2,8.1,0.28,0.40,6.9,0.050,30.0,97.0,0.9951,3.26,0.44,10.1,6
3,7.2,0.23,0.32,8.5,0.058,47.0,186.0,0.9956,3.19,0.40,9.9,6
4,7.2,0.23,0.32,8.5,0.058,47.0,186.0,0.9956,3.19,0.40,9.9,6


In [15]:
white_wine.describe()

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality
count,4898.000000,4898.000000,4898.000000,4898.000000,4898.000000,4898.000000,4898.000000,4898.000000,4898.000000,4898.000000,4898.000000,4898.000000
mean,6.854788,0.278241,0.334192,6.391415,0.045772,35.308085,138.360657,0.994027,3.188267,0.489847,10.514267,5.877909
std,0.843868,0.100795,0.121020,5.072058,0.021848,17.007137,42.498065,0.002991,0.151001,0.114126,1.230621,0.885639
min,3.800000,0.080000,0.000000,0.600000,0.009000,2.000000,9.000000,0.987110,2.720000,0.220000,8.000000,3.000000
25%,6.300000,0.210000,0.270000,1.700000,0.036000,23.000000,108.000000,0.991723,3.090000,0.410000,9.500000,5.000000
50%,6.800000,0.260000,0.320000,5.200000,0.043000,34.000000,134.000000,0.993740,3.180000,0.470000,10.400000,6.000000
75%,7.300000,0.320000,0.390000,9.900000,0.050000,46.000000,167.000000,0.996100,3.280000,0.550000,11.400000,6.000000
max,14.200000,1.100000,1.660000,65.800000,0.346000,289.000000,440.000000,1.038980,3.820000,1.080000,14.200000,9.000000


Wine quality needs to be predicted and the range looks correct. In its simplest form, the problem could be seen as a regression task, for example by predicting `quality` as a floating point number, which can be rounded to the closest integer, however, the scope of the exercise is predicting `accuracy`, and therefore this should be turned into a classification problem. The most natural approach to do so would be to using the `softmax` function during training, to return the probabilities of each observation belonging to a certain quality label. The problem could be simplified further by performing featuring engineering on the target variable to divide it into two or three ranges of wine quality (e.g., low, medium, high).

In [16]:
# Turn the data into a np.ndarray format for easy processing
# The features' names are not needed for this exercise
white_wine_array = np.array(white_wine)
X, y = white_wine_array[:, 0:11], white_wine_array[:, -1]

In [17]:
X.shape

(4898, 11)

In [18]:
y.shape

(4898,)

In [19]:
Xtrain, Xtest, ytrain, ytest = train_test_split(X, y, test_size=0.25, seed=42)

In [20]:
print(f"SHAPES: Xtrain: {Xtrain.shape}, Xtest: {Xtest.shape}, ytrain: {ytrain.shape}, ytest: {ytest.shape}")

SHAPES: Xtrain: (3673, 11), Xtest: (1225, 11), ytrain: (3673,), ytest: (1225,)


In [21]:
scaled_Xtrain, scaled_Xtest, std, mu = standardise_training_data(Xtrain, Xtest)

## Modelling

In [22]:
methods_list = ["lstsq", "solve", "inv"]

In [32]:
results = run_multiple_experiments(scaled_Xtrain, ytrain, scaled_Xtest, ytest, methods_list, degree=1)

In [33]:
results

,method,time,train_rss,test_rss
1,solve,0.000181,2862.867745,928.648536
2,inv,0.000186,2862.867745,928.648536
0,lstsq,0.000561,2862.867745,928.648536


Expectedly, test and train RSS losses are the same across methods, since they are all solving the same normal equations for a
linear least-squares model, which has a closed solution. Therefore, unless numerical issues arise, the resulting weight vectors and predictions align.
In terms of computation time, `solve` and `inv` are almost identical in this small problem. Both rely on direct linear-algebra factorisations of the normal equation matrix. The `lstsq` solver, however, is noticeably slower. This is expected because `lstsq` uses a more numerically robust decomposition (`QR` or `SVD`), which incurs extra computational cost. In exchange for this cost, it is usually more stable for ill-conditioned or high-degree polynomial models. However, in this dataset and polynomial degree, the system is well-conditioned enough that all solvers behave the same in RSS loss.

## REFERENCES

Cortez, P., Cerdeira, A., Almeida, F., Matos, T., & Reis, J. (2009). Wine Quality [Dataset]. UCI Machine Learning Repository. https://doi.org/10.24432/C56S3T.